In [9]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# -------------------------
# Customers Table
# -------------------------
customers = pd.DataFrame({
    "customer_id": [f"C{i:03}" for i in range(1,9)],
    "name": ["Alice","Bob","Charlie","David","Emma","Frank","Grace","Helen"],
    "email": [
        "alice@mail.com","bob@mail.com","charlie@mail.com","david@mail.com",
        "emma@mail.com","frank@mail.com","grace@mail.com","helen@mail.com"
    ],
    "city": ["New York","London","Paris","Berlin","Tokyo","Madrid","Rome","Toronto"],
    "signup_date": pd.date_range(start="2023-01-01", periods=8)
})

# -------------------------
# Movies Table
# -------------------------
movies = pd.DataFrame({
    "movie_id": [f"M{i:03}" for i in range(1,11)],
    "title": [
        "Inception","Titanic","Avengers","Interstellar","Joker",
        "Frozen","Matrix","Gladiator","Toy Story","The Godfather"
    ],
    "genre": [
        "Sci-Fi","Romance","Action","Sci-Fi","Drama",
        "Animation","Sci-Fi","Action","Animation","Crime"
    ],
    "release_year":[2010,1997,2012,2014,2019,2013,1999,2000,1995,1972],
    "rating":[5,4,5,5,4,4,5,4,4,5]
})

# -------------------------
# Rentals Table
# -------------------------
np.random.seed(1)

rentals = pd.DataFrame({
    "rental_id":[f"R{i:03}" for i in range(1,26)],
    "customer_id":np.random.choice(customers.customer_id,25),
    "movie_id":np.random.choice(movies.movie_id,25),
    "rental_date":[datetime(2024,1,1)+timedelta(days=int(x)) for x in np.random.randint(0,60,25)],
    "return_date":[datetime(2024,1,5)+timedelta(days=int(x)) for x in np.random.randint(0,60,25)],
    "price":np.random.randint(3,10,25)
})

customers.head(), movies.head(), rentals.head()

customer = "C001"

movies_by_customer = rentals.merge(movies,on="movie_id")
movies_by_customer = movies_by_customer[movies_by_customer.customer_id == customer]

movies_by_customer[["title","genre","rental_date","price"]]

top_movies = rentals.groupby("movie_id").size().reset_index(name="rental_count")

top_movies = top_movies.merge(movies,on="movie_id")

top_movies.sort_values("rental_count",ascending=False).head(5)[["title","rental_count"]]

revenue_genre = rentals.merge(movies,on="movie_id")

revenue_genre = revenue_genre.groupby("genre")["price"].sum().reset_index()

revenue_genre.sort_values("price",ascending=False)

merged = rentals.merge(movies,on="movie_id")

low_rating_customers = merged[merged.rating < 3].customer_id.unique()

good_customers = customers[~customers.customer_id.isin(low_rating_customers)]

good_customers[["customer_id","name"]]

denormalized = rentals.merge(customers,on="customer_id").merge(movies,on="movie_id")

denormalized.head()

,rental_id,customer_id,movie_id,rental_date,return_date,price,name,email,city,signup_date,title,genre,release_year,rating
0,R001,C006,M003,2024-01-22,2024-01-30,5,Frank,frank@mail.com,Madrid,2023-01-06,Avengers,Action,2012,5
1,R002,C004,M005,2024-02-19,2024-02-14,9,David,david@mail.com,Berlin,2023-01-04,Joker,Drama,2019,4
2,R003,C005,M008,2024-02-27,2024-01-27,9,Emma,emma@mail.com,Tokyo,2023-01-05,Gladiator,Action,2000,4
3,R004,C001,M008,2024-01-04,2024-01-14,3,Alice,alice@mail.com,New York,2023-01-01,Gladiator,Action,2000,4
4,R005,C008,M010,2024-01-05,2024-01-08,3,Helen,helen@mail.com,Toronto,2023-01-08,The Godfather,Crime,1972,5


Denormalized Table Issues

When joining all tables into one:

Customer information (name, email, city) repeats for every rental.

Movie information (title, genre, rating) repeats every time the movie is rented.

If a movie title changes, many rows must be updated.

Why Normalization is Better

Normalized tables:

Reduce redundancy

Improve consistency

Make updates easier

Save storage space

The relational schema separates customers, movies, and rentals, and relationships are maintained through foreign keys.

In [14]:
documents = []

for _,cust in customers.iterrows():

    cust_rentals = rentals[rentals.customer_id == cust.customer_id]

    rental_list = []

    for _,rent in cust_rentals.iterrows():

        movie = movies[movies.movie_id == rent.movie_id].iloc[0]

        rental_list.append({
            "movie_id":movie.movie_id,
            "title":movie.title,
            "genre":movie.genre,
            "rating":movie.rating,
            "rental_date":rent.rental_date,
            "price":rent.price
        })

    documents.append({
        "customer_id":cust.customer_id,
        "name":cust.name,
        "city":cust.city,
        "rentals":rental_list
    })

documents[:2]



customer = "C001"

for doc in documents:
    if doc["customer_id"] == customer:
        print([r["title"] for r in doc["rentals"]])

from collections import Counter

counter = Counter()

for doc in documents:
    for r in doc["rentals"]:
        counter[r["title"]] += 1

counter.most_common(5)

genre_revenue = {}

for doc in documents:
    for r in doc["rentals"]:
        genre_revenue[r["genre"]] = genre_revenue.get(r["genre"],0) + r["price"]

genre_revenue

good_customers = []

for doc in documents:

    if all(r["rating"] >= 3 for r in doc["rentals"]):
        good_customers.append(doc["name"])

good_customers

['Gladiator', 'The Godfather', 'The Godfather']


[0, 1, 2, 3, 4, 5, 6, 7]

Relational Queries

Advantages:

Structured joins

Easy aggregation

Efficient indexing

Disadvantages:

Multiple joins needed for related data.

Document Queries

Advantages:

Customer and rentals stored together.

Retrieving one customer's history is very fast.

Disadvantages:

Aggregations across all documents are harder.

Data duplication occurs.

Clear Advantage of Document Model

Queries that retrieve all information about one entity (like a customer's rental history) are simpler and faster because everything is embedded in one document.

In [19]:
nodes = {}

# customers
for _,c in customers.iterrows():
    nodes[c.customer_id] = {"type":"customer","name":c.name}

# movies
for _,m in movies.iterrows():
    nodes[m.movie_id] = {"type":"movie","title":m.title}

# genres
for g in movies.genre.unique():
    nodes[g] = {"type":"genre","name":g}

edges = []

for _,r in rentals.iterrows():
    edges.append((r.customer_id,"rented",r.movie_id))

for _,m in movies.iterrows():
    edges.append((m.movie_id,"belongs_to",m.genre))


def movies_rented(customer):
    return [e[2] for e in edges if e[0]==customer and e[1]=="rented"]

def preferred_genres(customer):

    movies_list = movies_rented(customer)

    genres = []

    for m in movies_list:
        for e in edges:
            if e[0]==m and e[1]=="belongs_to":
                genres.append(e[2])

    return pd.Series(genres).value_counts()


def similar_customers(customer):

    genres = preferred_genres(customer).index.tolist()

    similar = set()

    for g in genres:
        for e in edges:
            if e[1]=="belongs_to" and e[2]==g:
                movie = e[0]

                for r in edges:
                    if r[1]=="rented" and r[2]==movie:
                        similar.add(r[0])

    similar.discard(customer)

    return similar

def recommend(customer):

    genres = preferred_genres(customer).index.tolist()

    rented = set(movies_rented(customer))

    rec = set()

    for g in genres:
        for e in edges:
            if e[1]=="belongs_to" and e[2]==g:
                if e[0] not in rented:
                    rec.add(e[0])

    return rec

Graph databases are ideal for recommendation systems because they represent relationships directly as edges.

The recommendation query requires:

Traversing from customer → rented movies

From movies → genres

Finding other movies in those genres

Filtering movies the customer has not rented

In SQL, this requires multiple joins and complex filtering.
In document databases, relationships across documents require scanning many records.

In contrast, graph traversal naturally follows edges and makes multi-hop queries efficient.

In [27]:
import random

n = 5000

transactions = pd.DataFrame({
    "transaction_id":[f"T{i:05}" for i in range(n)],
    "customer_id":np.random.choice(customers.customer_id,n),
    "product_id":[f"P{random.randint(1,100)}" for _ in range(n)],
    "amount":np.random.uniform(5,500,n),
    "payment_method":np.random.choice(["card","paypal","crypto"],n),
    "timestamp":pd.date_range("2024-01-01",periods=n,freq="h"),
    "status":np.random.choice(["completed","pending","refunded"],n)
})

transactions.head()

# Point query
transactions[transactions.transaction_id=="T00010"]

# Insert
new_row = pd.DataFrame([{
    "transaction_id":"T99999",
    "customer_id":"C001",
    "product_id":"P10",
    "amount":120,
    "payment_method":"card",
    "timestamp":pd.Timestamp.now(),
    "status":"pending"
}])

transactions = pd.concat([transactions,new_row],ignore_index=True)

# Update
transactions.loc[transactions.transaction_id=="T99999","status"]="completed"

# Consistency check
assert (transactions.amount > 0).all()
assert transactions.status.isin(["completed","pending","refunded"]).all()

transactions.groupby("payment_method")["amount"].sum()

transactions["month"] = transactions.timestamp.dt.to_period("M")

transactions.groupby("month")["amount"].mean()

transactions["month"] = transactions.timestamp.dt.to_period("M")

transactions.groupby("month")["amount"].mean()

transactions.groupby("customer_id")["amount"].sum().sort_values(ascending=False).head(10)

refund_rate = (transactions.status=="refunded").mean()*100
refund_rate



np.float64(32.33353329334133)

Row-major storage (OLTP)

Stores entire rows together.

Efficient for point queries and updates.

Example: retrieving one transaction requires reading only one row.

Column-major storage (OLAP)

Stores columns separately.

Efficient for aggregation queries.

Example: computing revenue only scans the amount column.

Access Pattern

OLTP → frequent inserts, updates, and single-row lookups.

OLAP → large scans, aggregations, and analytical queries.

Column storage minimizes I/O for analytical queries.